# Scania APS - Exploratory Data Analysis
**Arkon Manufacturing AI | Module: ML Classification | Department: Truck Fleet**

APS Failure at Scania Trucks dataset - real data donated by Scania CV AB to UCI ML Repository.

**Business context:** The Air Pressure System (APS) generates compressed air for brakes and suspension.
An APS failure can cause a truck to break down on the road - extremely expensive and dangerous.

**Cost structure (from original Scania competition):**
- False Positive (predict failure, no failure): **cost = 10** (unnecessary inspection)
- False Negative (miss a failure): **cost = 500** (breakdown on road)

→ Missing a real failure is **50x more expensive** than a false alarm.

**Goal:** Understand class imbalance, missing values, and feature distributions.

In [ ]:
import sys
from pathlib import Path

# Add notebooks/utils to path (works on Mac, Windows, Linux)
_nb_root = Path('..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)

print('arkon_utils loaded ✓')

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
ASSETS = 'ml'


## 2. Load Data

In [ ]:
# Load training and test sets - note na_values='na' to handle Scania missing value format
df_train = pd.read_csv(
    DATA_DIR / 'aps_failure_training_set.csv',
    na_values='na',
    skiprows=20   # first 20 rows are description text
)

df_test = pd.read_csv(
    DATA_DIR / 'aps_failure_test_set.csv',
    na_values='na',
    skiprows=20
)

print(f'Train shape: {df_train.shape}')
print(f'Test shape:  {df_test.shape}')

## 3. First Look

In [ ]:
# Show first 5 rows - features are anonymised (aa_000, ab_000, etc.)
df_train.head()

In [ ]:
# Check data types - all features should be float, target is string (pos/neg)
print(df_train.dtypes.value_counts())
print(f'\nTarget column unique values: {df_train["class"].unique()}')

## 4. Class Balance - Key Issue

In [ ]:
# Count positive (APS failure) vs negative (no failure) samples - expect severe imbalance
class_counts = df_train['class'].value_counts()
class_pct    = df_train['class'].value_counts(normalize=True) * 100

print('Class distribution (training set):')
for cls in class_counts.index:
    print(f'  {cls}: {class_counts[cls]:,} samples ({class_pct[cls]:.2f}%)')

print(f'\nImbalance ratio: 1:{class_counts["neg"] // class_counts["pos"]} (neg:pos)')

In [ ]:
# Visualise class imbalance - shows why accuracy alone is a misleading metric here
fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='white')
ax.set_xticklabels(['Negative (no failure)', 'Positive (APS failure)'], rotation=0)
ax.set_ylabel('Number of samples')
ax.set_title('Class Imbalance - Scania APS Training Set')

# Add count labels on bars
for i, v in enumerate(class_counts):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
save_figure(fig, 'scania_eda_class_imbalance', subfolder=ASSETS)
plt.show()

## 5. Missing Values Analysis

In [ ]:
# Calculate percentage of missing values per feature column
feature_cols = [c for c in df_train.columns if c != 'class']
missing_pct = df_train[feature_cols].isnull().mean() * 100
missing_pct = missing_pct.sort_values(ascending=False)

print(f'Features with >0% missing: {(missing_pct > 0).sum()}')
print(f'Features with >50% missing: {(missing_pct > 50).sum()}')
print(f'Features with >90% missing: {(missing_pct > 90).sum()}')
print(f'\nTop 10 most missing:')
print(missing_pct.head(10).round(1))

In [ ]:
# Plot histogram of missing value percentages across all features
fig, ax = plt.subplots(figsize=(10, 4))
missing_pct[missing_pct > 0].plot(kind='hist', bins=30, ax=ax,
                                   color='steelblue', edgecolor='white')
ax.axvline(50, color='red', linestyle='--', label='50% threshold')
ax.set_xlabel('Missing Values (%)')
ax.set_ylabel('Number of Features')
ax.set_title('Distribution of Missing Values Across Features')
ax.legend()
plt.tight_layout()
save_figure(fig, 'scania_eda_missing_values', subfolder=ASSETS)
plt.show()

## 6. Feature Statistics

In [ ]:
# Show descriptive statistics for first 10 features - understand value ranges
df_train[feature_cols[:10]].describe().round(2)

## 7. Feature Distribution by Class

In [ ]:
# Compare feature distributions between pos and neg classes for top informative features
# Select features with lowest missing rate for this visualisation
low_missing = missing_pct[missing_pct < 5].index.tolist()[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(low_missing):
    df_train[df_train['class'] == 'neg'][feat].dropna().plot(
        kind='hist', ax=axes[i], alpha=0.5, bins=30,
        color='steelblue', label='neg', density=True
    )
    df_train[df_train['class'] == 'pos'][feat].dropna().plot(
        kind='hist', ax=axes[i], alpha=0.5, bins=30,
        color='tomato', label='pos', density=True
    )
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('')

plt.suptitle('Feature Distributions by Class (pos=APS failure, neg=ok)', fontsize=12)
plt.tight_layout()
save_figure(fig, 'scania_eda_class_imbalance', subfolder=ASSETS)
plt.show()

## 8. Cost Analysis

In [ ]:
# Calculate what a naive 'always predict negative' model would cost
# This shows why accuracy is misleading and why we need recall-focused evaluation
n_pos = class_counts['pos']  # actual failures
n_neg = class_counts['neg']  # actual non-failures

COST_FP = 10    # false positive: unnecessary inspection
COST_FN = 500   # false negative: breakdown on road

# Naive model: always predict 'no failure'
naive_fn_cost = n_pos * COST_FN  # all real failures missed
naive_fp_cost = 0
naive_accuracy = n_neg / (n_pos + n_neg) * 100

print('Naive model (always predict NEG):')
print(f'  Accuracy:      {naive_accuracy:.1f}%  ← looks good but misleading')
print(f'  False Negatives: {n_pos} missed failures')
print(f'  Total cost:    {naive_fn_cost:,} units')
print(f'\nOur model goal: minimise cost = {COST_FP}×FP + {COST_FN}×FN')

## 9. Correlation Among Features (Sample)

In [ ]:
# Plot correlation matrix for 20 features with least missing data
sample_feats = missing_pct[missing_pct < 5].index.tolist()[:20]
corr = df_train[sample_feats].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax,
            linewidths=0.3, annot=False)
ax.set_title('Feature Correlation Matrix (20 least-missing features)')
plt.tight_layout()
save_figure(fig, 'scania_eda_missing_values', subfolder=ASSETS)
plt.show()

## 10. Key EDA Findings

| Finding | Detail | Action in Preprocessing |
|---------|--------|--------------------------|
| Severe class imbalance | ~2% positive | Use SMOTE + class weights |
| Missing values everywhere | Up to 90%+ in some features | Median imputation |
| 171 anonymised features | No domain names | Feature selection by importance |
| Cost-sensitive problem | FN = 50× FP | Optimise threshold, not accuracy |
| Target is string | 'pos' / 'neg' | Encode to 1 / 0 |

**Next step:** `02_scania_preprocessing.ipynb` - impute, encode, SMOTE, feature selection.